In [ ]:

## Perform 4D segmentation volume by volume
## this script is used to predict 4D data
## the input is a 4D image, and the output is a 4D segmentation

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import SimpleITK as sitk


from predict_single_LVSA import  predict


In [6]:
def convert4dto3d(input_image_path, save_format="temp_{}.nii.gz"):
    ## load data from simple itk to check whether the input data is 3D or 4D data
    ## save data into individual 3D volumes under the same folder with the name of temp_{}.nii.gz
    ## return number of 3D volumes, and saving path list
    sitk_img =  sitk.ReadImage(input_image_path)
    n_dim = sitk_img.GetDimension()
    save_path_list = []
    i=0
    if n_dim == 3:
        print('3D data')
        return 1, [input_image_path]
    elif n_dim == 4:
        print('4D data')
        print('The input data is 4D data, we will get each 3D volume separately')
        ## save data into individual 3D volume and predict each 3D volume separately
        # Get the size of the 4th dimension
        size_4th_dim = sitk_img.GetSize()[3]
        # Loop over the 4th dimension and extract each 3D image
        for i in range(size_4th_dim):
            # Extract the i-th 3D image
            slice_image = sitk.Extract(sitk_img, (sitk_img.GetSize()[0], sitk_img.GetSize()[1], sitk_img.GetSize()[2], 0), (0, 0, 0, i))
            
            parent_dir= os.path.dirname(input_image_path)
            print (slice_image.GetSize())
            # Save the 3D image as a separate NIfTI file
            save_path = os.path.join(parent_dir,save_format.format(i))
            sitk.WriteImage(slice_image,save_path)
            save_path_list.append(save_path)
            print('Success: The {}th 3D volume is saved'.format(i))
    else:
        raise NotImplementedError
    return i, save_path_list


In [7]:
## merge 3D volume into 4D volume
def merge_3D_to_4D(input_image_path_list, output_image_path,reference_4D_image_path):
    # Load the first 3D NIfTI file to use as a template for the output 4D file
    template_image = sitk.ReadImage(reference_4D_image_path)
    image_list = []
    
    for i, input_file in enumerate(input_image_path_list):
        print (f'loading {str(i)}-th image, path: {input_file} ')
        # Load the i-th 3D NIfTI file
        input_image = sitk.ReadImage(input_file)
        image_list.append(input_image)

    output_image = sitk.JoinSeries(image_list) #join list of three images
    print (output_image.GetSize())
    output_image.CopyInformation(template_image)
    # Save the output 4D NIfTI file
    sitk.WriteImage(output_image, output_image_path)
    print (f'Success! saving 4D image to {output_image_path}')



In [8]:

## config
gpu_id=0
device = torch.device("cuda:{}".format(gpu_id) if (torch.cuda.is_available()) else "cpu")
if torch.cuda.is_available():
    torch.cuda.set_device(gpu_id)
os.environ["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
model_path= '/home/engs2522/project/shared_projects/Cardiac_Multi_view_segmentation/checkpoints/LVSA/Unet_adv_chain.pth'
## 4d images
input_image_path = '/home/engs2522/project/shared_projects/Cardiac_Multi_view_segmentation/test_results/LVSA/Ahbi/sa.nii.gz'

## first convert 4D image to 3D image
save_format="temp_{}.nii.gz"
n_volumes, path_list = convert4dto3d(input_image_path, save_format=save_format)

4D data
The input data is 4D data, we will get each 3D volume separately
(210, 208, 11)
Success: The 0th 3D volume is saved
(210, 208, 11)
Success: The 1th 3D volume is saved
(210, 208, 11)
Success: The 2th 3D volume is saved
(210, 208, 11)
Success: The 3th 3D volume is saved
(210, 208, 11)
Success: The 4th 3D volume is saved
(210, 208, 11)
Success: The 5th 3D volume is saved
(210, 208, 11)
Success: The 6th 3D volume is saved
(210, 208, 11)
Success: The 7th 3D volume is saved
(210, 208, 11)
Success: The 8th 3D volume is saved
(210, 208, 11)
Success: The 9th 3D volume is saved
(210, 208, 11)
Success: The 10th 3D volume is saved
(210, 208, 11)
Success: The 11th 3D volume is saved
(210, 208, 11)
Success: The 12th 3D volume is saved
(210, 208, 11)
Success: The 13th 3D volume is saved
(210, 208, 11)
Success: The 14th 3D volume is saved
(210, 208, 11)
Success: The 15th 3D volume is saved
(210, 208, 11)
Success: The 16th 3D volume is saved
(210, 208, 11)
Success: The 17th 3D volume is saved
(

# standard prediction

In [10]:

## conduct prediction on each 3D volume
torch.cuda.empty_cache()
save_pred_path_list = []
for image_path in path_list:
    save_pred_path = image_path.replace('.nii.gz','_pred.nii.gz')
    save_pred_path_list.append(save_pred_path)
    model,im,pred = predict (model_path, image_path,save_pred_path=save_pred_path, batch_size=1, crop_size=192,if_resample=True,if_z_score=True,gpu_id=gpu_id)     



<------Loading model-------->
init UNet_64
load params from  /home/engs2522/project/shared_projects/Cardiac_Multi_view_segmentation/checkpoints/LVSA/Unet_adv_chain.pth


/home/engs2522/project/shared_projects/Cardiac_Multi_view_segmentation/model/init_weight.py:31: UserWarning: nn.init.kaiming_normal is now deprecated in favor of nn.init.kaiming_normal_.
  init.kaiming_normal(m.weight.data, a=0, mode='fan_in')
/home/engs2522/project/shared_projects/Cardiac_Multi_view_segmentation/model/init_weight.py:35: UserWarning: nn.init.normal is now deprecated in favor of nn.init.normal_.
  init.normal(m.weight.data, 1.0, 0.02)
/home/engs2522/project/shared_projects/Cardiac_Multi_view_segmentation/model/init_weight.py:36: UserWarning: nn.init.constant is now deprecated in favor of nn.init.constant_.
  init.constant(m.bias.data, 0.0)


<------Loading data-------->
<------Preprocessing data-------->
<------Batchwise prediction-------->
<------Recover image information-------->
Saving segmentation to /home/engs2522/project/shared_projects/Cardiac_Multi_view_segmentation/test_results/LVSA/Ahbi/temp_0_pred.nii.gz
<------Loading model-------->
init UNet_64
load params from  /home/engs2522/project/shared_projects/Cardiac_Multi_view_segmentation/checkpoints/LVSA/Unet_adv_chain.pth
<------Loading data-------->
<------Preprocessing data-------->
<------Batchwise prediction-------->
<------Recover image information-------->
Saving segmentation to /home/engs2522/project/shared_projects/Cardiac_Multi_view_segmentation/test_results/LVSA/Ahbi/temp_1_pred.nii.gz
<------Loading model-------->
init UNet_64
load params from  /home/engs2522/project/shared_projects/Cardiac_Multi_view_segmentation/checkpoints/LVSA/Unet_adv_chain.pth
<------Loading data-------->
<------Preprocessing data-------->
<------Batchwise prediction-------->
<----

## Stack Prediction back to 4D space

In [11]:


template_image = sitk.ReadImage(input_image_path)
parent_dir= os.path.dirname(input_image_path)
output_image_path = os.path.join(parent_dir,'pred_4D.nii.gz')
merge_3D_to_4D(save_pred_path_list, output_image_path,reference_4D_image_path=input_image_path) 

loading 0-th image, path: /home/engs2522/project/shared_projects/Cardiac_Multi_view_segmentation/test_results/LVSA/Ahbi/temp_0_pred.nii.gz 
loading 1-th image, path: /home/engs2522/project/shared_projects/Cardiac_Multi_view_segmentation/test_results/LVSA/Ahbi/temp_1_pred.nii.gz 
loading 2-th image, path: /home/engs2522/project/shared_projects/Cardiac_Multi_view_segmentation/test_results/LVSA/Ahbi/temp_2_pred.nii.gz 
loading 3-th image, path: /home/engs2522/project/shared_projects/Cardiac_Multi_view_segmentation/test_results/LVSA/Ahbi/temp_3_pred.nii.gz 
loading 4-th image, path: /home/engs2522/project/shared_projects/Cardiac_Multi_view_segmentation/test_results/LVSA/Ahbi/temp_4_pred.nii.gz 
loading 5-th image, path: /home/engs2522/project/shared_projects/Cardiac_Multi_view_segmentation/test_results/LVSA/Ahbi/temp_5_pred.nii.gz 
loading 6-th image, path: /home/engs2522/project/shared_projects/Cardiac_Multi_view_segmentation/test_results/LVSA/Ahbi/temp_6_pred.nii.gz 
loading 7-th image, 